In [3]:
import csv
import os
from datetime import datetime

try:
    import matplotlib.pyplot as plt
    have_matplotlib = True
except ImportError:
    have_matplotlib = False

DATA_FOLDER = "private_data"
DATA_FILE = DATA_FOLDER + "/amr_records.csv"
OUTPUT_FOLDER = "outputs"
MIN_TESTED_FOR_CHART = 3

antibiotics = [
    "ampicillin", "amoxicillin_clavulanate",
    "ceftriaxone", "cefuroxime", "cefoxitin", "ceftazidime", "cefepime",
    "levofloxacin", "ciprofloxacin", "norfloxacin",
    "gentamicin", "amikacin",
    "imipenem", "meropenem", "ertapenem",
    "piperacillin_tazobactam",
    "cotrimoxazole",
    "nitrofurantoin",
    "vancomycin", "teicoplanin",
    "doxycycline",
    "azithromycin",
    "colistin",
    "aztreonam",
    "linezolid",
]


antibiotic_class = {
    "ampicillin": "penicillins",
    "amoxicillin_clavulanate": "penicillin + beta-lactamase inhibitor",
    "ceftriaxone": "cephalosporins",
    "cefuroxime": "cephalosporins",
    "cefoxitin": "cephalosporins",
    "ceftazidime": "cephalosporins",
    "cefepime": "cephalosporins",
    "levofloxacin": "fluoroquinolones",
    "ciprofloxacin": "fluoroquinolones",
    "norfloxacin": "fluoroquinolones",
    "gentamicin": "aminoglycosides",
    "amikacin": "aminoglycosides",
    "imipenem": "carbapenems",
    "meropenem": "carbapenems",
    "ertapenem": "carbapenems",
    "piperacillin_tazobactam": "penicillin + beta-lactamase inhibitor",
    "cotrimoxazole": "folate pathway inhibitors",
    "nitrofurantoin": "nitrofurans",
    "vancomycin": "glycopeptides",
    "teicoplanin": "glycopeptides",
    "doxycycline": "tetracyclines",
    "azithromycin": "macrolides",
    "colistin": "polymyxins",
    "aztreonam": "monobactams",
    "linezolid": "oxazolidinones",
}

genders = ["Male", "Female", "Not recorded"]


sample_types = ["Urine", "Throat swab", "Pus", "Sputum", "Wound swab", "Nasal swab", "Stool", "Other"]


organisms = [
    "Escherichia coli",
    "Klebsiella spp.",
    "Pseudomonas spp.",
    "Staphylococcus aureus",
    "Streptococcus spp.",
    "Proteus spp.",
    "Enterococcus spp.",
    "Acinetobacter spp.",
    "Citrobacter spp.",
    "Enterobacter spp.",
]


gram_negative_panel = [
    "ampicillin", "amoxicillin_clavulanate",
    "ceftriaxone", "cefuroxime", "ceftazidime", "cefepime",
    "levofloxacin", "ciprofloxacin", "norfloxacin",
    "gentamicin", "amikacin",
    "imipenem", "meropenem", "ertapenem",
    "piperacillin_tazobactam",
    "cotrimoxazole",
    "nitrofurantoin",
    "aztreonam",
]

nonfermenter_panel = [
    "ceftazidime", "cefepime",
    "levofloxacin", "ciprofloxacin",
    "gentamicin", "amikacin",
    "imipenem", "meropenem",
    "piperacillin_tazobactam",
    "colistin",
    "cotrimoxazole",
]

staph_panel = [
    "ceftriaxone", "cefuroxime", "cefoxitin",
    "levofloxacin", "ciprofloxacin",
    "vancomycin", "teicoplanin", "linezolid",
    "doxycycline", "azithromycin", "cotrimoxazole",
]

strep_panel = [
    "ampicillin", "ceftriaxone",
    "levofloxacin",
    "vancomycin", "linezolid", "azithromycin",
]

enterococcus_panel = [
    "ampicillin",
    "vancomycin", "teicoplanin", "linezolid",
    "levofloxacin",
    "nitrofurantoin",
    "gentamicin",
]

panel = {
    "Escherichia coli": gram_negative_panel,
    "Klebsiella spp.": gram_negative_panel,
    "Proteus spp.": gram_negative_panel,
    "Citrobacter spp.": gram_negative_panel,
    "Enterobacter spp.": gram_negative_panel,
    "Pseudomonas spp.": nonfermenter_panel,
    "Acinetobacter spp.": nonfermenter_panel,
    "Staphylococcus aureus": staph_panel,
    "Streptococcus spp.": strep_panel,
    "Enterococcus spp.": enterococcus_panel,
}


esbl_relevant_organisms = ["Escherichia coli", "Klebsiella spp.", "Proteus spp.", "Citrobacter spp.", "Enterobacter spp."]

column_names = ["patient_id", "gender", "sample_date", "sample_type", "organism", "esbl_reported"] + antibiotics



def load_records():
    records = []
    if os.path.exists(DATA_FILE):
        file = open(DATA_FILE, newline="")
        reader = csv.DictReader(file)
        for row in reader:
            records.append(row)
        file.close()
    return records


def save_records(records):
    if not os.path.exists(DATA_FOLDER):
        os.makedirs(DATA_FOLDER)
    write_csv(DATA_FILE, column_names, records)


def write_csv(filename, columns, rows):
    file = open(filename, "w", newline="")
    writer = csv.DictWriter(file, fieldnames=columns)
    writer.writeheader()
    for row in rows:
        writer.writerow(row)
    file.close()


def ask_text(question):
    while True:
        answer = input(question)
        answer = answer.strip()
        if answer != "":
            return answer
        print("This cannot be empty, try again.")


def ask_choice(question, options):
    print(question)
    i = 0
    while i < len(options):
        print("  " + str(i + 1) + ". " + options[i])
        i = i + 1
    while True:
        answer = input("Enter the number: ")
        answer = answer.strip()
        if answer.isdigit():
            number = int(answer)
            if number >= 1 and number <= len(options):
                return options[number - 1]
        print("Please type one of the numbers shown above.")


def ask_yes_no(question):
    while True:
        answer = input(question + " (y/n): ")
        answer = answer.strip().lower()
        if answer == "y" or answer == "yes":
            return True
        if answer == "n" or answer == "no":
            return False
        print("Please type y or n.")


def ask_date():
    while True:
        text = input("Sample date (DD/MM/YYYY, or just press Enter for today): ")
        text = text.strip()
        if text == "":
            today = datetime.today()
            return today.strftime("%Y-%m-%d")
        try:
            entered_date = datetime.strptime(text, "%d/%m/%Y")
            return entered_date.strftime("%Y-%m-%d")
        except ValueError:
            print("That does not look like a real date. Example: 25/04/2026")


def ask_result(antibiotic):
    nice_name = antibiotic.replace("_", " ")
    while True:
        answer = input("  " + nice_name + " (S / I / R, or Enter if not tested): ")
        answer = answer.strip().lower()
        if answer == "":
            return ""
        if answer == "s" or answer == "sensitive":
            return "S"
        if answer == "i" or answer == "intermediate":
            return "I"
        if answer == "r" or answer == "resistant":
            return "R"
        print("  Please type S, I or R, or just press Enter to skip.")



def classes_resistant(record):

    found_classes = []
    for antibiotic in antibiotics:
        if record[antibiotic] == "R":
            this_class = antibiotic_class[antibiotic]
            if this_class not in found_classes:
                found_classes.append(this_class)
    return found_classes


def is_mdr(record):
    resistant_classes = classes_resistant(record)
    if len(resistant_classes) >= 3:
        return True
    return False


def is_carbapenem_resistant(record):
    if record["imipenem"] == "R":
        return True
    if record["meropenem"] == "R":
        return True
    if record["ertapenem"] == "R":
        return True
    return False


def staph_type(record):

    if record["organism"] == "Staphylococcus aureus":
        if record["cefoxitin"] == "S":
            return "MSSA"
        if record["cefoxitin"] == "R":
            return "MRSA"
    return ""


def flag_text(record):
    flags = []
    if record["esbl_reported"] == "Yes":
        flags.append("ESBL")
    if is_mdr(record):
        n = len(classes_resistant(record))
        flags.append("MDR (" + str(n) + " classes)")
    if is_carbapenem_resistant(record):
        flags.append("Carbapenem-resistant")
    staph_flag = staph_type(record)
    if staph_flag != "":
        flags.append(staph_flag)
    if len(flags) == 0:
        return "none"
    text = ""
    for f in flags:
        if text == "":
            text = f
        else:
            text = text + ", " + f
    return text



def add_isolate(records):
    print("")
    print("--- New isolate ---")
    record = {}
    record["patient_id"] = ask_text("Lab ID or code (NOT the patient's name): ")
    record["gender"] = ask_choice("Gender:", genders)
    record["sample_date"] = ask_date()
    record["sample_type"] = ask_choice("Sample type:", sample_types)

    organism_choice = ask_choice("Organism:", organisms + ["Other (type the name)"])
    if organism_choice.startswith("Other"):
        organism_choice = ask_text("Type the organism name: ")
    record["organism"] = organism_choice


    for old_record in records:
        same_id = old_record["patient_id"] == record["patient_id"]
        same_date = old_record["sample_date"] == record["sample_date"]
        same_sample = old_record["sample_type"] == record["sample_type"]
        same_organism = old_record["organism"].lower() == record["organism"].lower()
        if same_id and same_date and same_sample and same_organism:
            print("")
            print("This isolate (same ID, date, sample and organism) is already in the file. Nothing added.")
            return


    record["esbl_reported"] = ""
    if record["organism"] in esbl_relevant_organisms:
        esbl_yes = ask_yes_no("ESBL producer?")
        if esbl_yes:
            record["esbl_reported"] = "Yes"
        else:
            record["esbl_reported"] = "No"


    for antibiotic in antibiotics:
        record[antibiotic] = ""

    if record["organism"] in panel:
        antibiotics_to_ask = panel[record["organism"]]
    else:
        antibiotics_to_ask = antibiotics

    print("Enter the sensitivity result for each antibiotic:")
    for antibiotic in antibiotics_to_ask:
        if antibiotic == "nitrofurantoin" and record["sample_type"] != "Urine":
            continue
        record[antibiotic] = ask_result(antibiotic)

    more_antibiotics = ask_yes_no("Enter results for any other antibiotics too?")
    if more_antibiotics:
        for antibiotic in antibiotics:
            if antibiotic not in antibiotics_to_ask:
                record[antibiotic] = ask_result(antibiotic)


    if record["esbl_reported"] == "Yes" and record["ceftriaxone"] == "S":
        print("")
        print("Warning: you marked this isolate ESBL but ceftriaxone came back sensitive. Please double check.")
        keep_it = ask_yes_no("Keep this entry anyway?")
        if not keep_it:
            print("Entry thrown away.")
            return

    records.append(record)
    save_records(records)
    print("")
    print("Saved. Flags for this isolate: " + flag_text(record))


def view_records(records):
    if len(records) == 0:
        print("")
        print("No isolates entered yet.")
        return
    print("")
    print("No.  ID              Date        Sample        Organism              Flags")
    i = 0
    while i < len(records):
        r = records[i]
        line = str(i + 1) + ".  " + r["patient_id"] + "  " + r["sample_date"] + "  " + r["sample_type"] + "  " + r["organism"] + "  " + flag_text(r)
        print(line)
        i = i + 1


def delete_isolate(records):
    view_records(records)
    if len(records) == 0:
        return
    answer = input("\nNumber of the isolate to delete (Enter to cancel): ")
    answer = answer.strip()
    if answer.isdigit():
        number = int(answer)
        if number >= 1 and number <= len(records):
            confirm = ask_yes_no("Really delete isolate " + answer + "?")
            if confirm:
                records.pop(number - 1)
                save_records(records)
                print("Deleted.")
                return
    print("Nothing deleted.")



def calculate_profile(records):
    n_esbl = 0
    n_mdr = 0
    n_carbapenem = 0
    n_mssa = 0
    n_mrsa = 0
    for r in records:
        if r["esbl_reported"] == "Yes":
            n_esbl = n_esbl + 1
        if is_mdr(r):
            n_mdr = n_mdr + 1
        if is_carbapenem_resistant(r):
            n_carbapenem = n_carbapenem + 1
        this_staph_type = staph_type(r)
        if this_staph_type == "MSSA":
            n_mssa = n_mssa + 1
        if this_staph_type == "MRSA":
            n_mrsa = n_mrsa + 1

    total = len(records)
    rows = []
    rows.append({"category": "ESBL-positive", "count": n_esbl, "percent_of_isolates": round(100 * n_esbl / total, 1)})
    rows.append({"category": "MDR (3+ classes)", "count": n_mdr, "percent_of_isolates": round(100 * n_mdr / total, 1)})
    rows.append({"category": "Carbapenem-resistant", "count": n_carbapenem, "percent_of_isolates": round(100 * n_carbapenem / total, 1)})
    rows.append({"category": "MSSA", "count": n_mssa, "percent_of_isolates": round(100 * n_mssa / total, 1)})
    rows.append({"category": "MRSA", "count": n_mrsa, "percent_of_isolates": round(100 * n_mrsa / total, 1)})
    return rows


def calculate_sensitivity(records):

    rows = []
    for sample in sample_types:
        for antibiotic in antibiotics:
            tested = 0
            sensitive = 0
            for r in records:
                if r["sample_type"] == sample and r[antibiotic] != "":
                    tested = tested + 1
                    if r[antibiotic] == "S":
                        sensitive = sensitive + 1
            if tested > 0:
                percent = round(100 * sensitive / tested, 1)
                rows.append({"sample_type": sample, "antibiotic": antibiotic, "n_tested": tested,
                             "n_sensitive": sensitive, "percent_sensitive": percent})
    return rows


def calculate_gender_counts(records):
    rows = []
    for sample in sample_types:
        males = 0
        females = 0
        unknown = 0
        for r in records:
            if r["sample_type"] == sample:
                if r["gender"] == "Male":
                    males = males + 1
                elif r["gender"] == "Female":
                    females = females + 1
                else:
                    unknown = unknown + 1
        total = males + females + unknown
        rows.append({"sample_type": sample, "total": total, "male": males, "female": females, "not_recorded": unknown})
    return rows


def calculate_organism_counts(records):
    names_seen = []
    for r in records:
        if r["organism"] not in names_seen:
            names_seen.append(r["organism"])
    rows = []
    for organism in names_seen:
        for sample in sample_types:
            count = 0
            for r in records:
                if r["organism"] == organism and r["sample_type"] == sample:
                    count = count + 1
            if count > 0:
                rows.append({"organism": organism, "sample_type": sample, "count": count})
    return rows


def show_summary(records):
    if len(records) == 0:
        print("")
        print("No isolates entered yet.")
        return

    print("")
    print("===== Summary of " + str(len(records)) + " isolates =====")

    print("")
    print("Resistance profile")
    for line in calculate_profile(records):
        print("  " + line["category"] + ": " + str(line["count"]) + " (" + str(line["percent_of_isolates"]) + "%)")

    print("")
    print("Isolates by sample type and gender")
    for line in calculate_gender_counts(records):
        if line["total"] > 0:
            print("  " + line["sample_type"] + " - total " + str(line["total"]) + ", male " + str(line["male"])
                  + ", female " + str(line["female"]) + ", not recorded " + str(line["not_recorded"]))

    print("")
    print("Antibiotic sensitivity by sample type (percent of tested isolates that were sensitive)")
    current_sample = ""
    for line in calculate_sensitivity(records):
        if line["sample_type"] != current_sample:
            current_sample = line["sample_type"]
            print("  " + current_sample)
        nice_name = line["antibiotic"].replace("_", " ")
        print("     " + nice_name + " - n=" + str(line["n_tested"]) + ", " + str(line["percent_sensitive"]) + "% sensitive")


def save_outputs(records):
    if len(records) == 0:
        print("")
        print("No isolates entered yet.")
        return

    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)

    sensitivity = calculate_sensitivity(records)

    write_csv(OUTPUT_FOLDER + "/resistance_profile.csv", ["category", "count", "percent_of_isolates"], calculate_profile(records))
    write_csv(OUTPUT_FOLDER + "/sensitivity_by_sample_type.csv",
              ["sample_type", "antibiotic", "n_tested", "n_sensitive", "percent_sensitive"], sensitivity)
    write_csv(OUTPUT_FOLDER + "/isolates_by_sample_and_gender.csv",
              ["sample_type", "total", "male", "female", "not_recorded"], calculate_gender_counts(records))
    write_csv(OUTPUT_FOLDER + "/organism_distribution.csv", ["organism", "sample_type", "count"], calculate_organism_counts(records))
    print("")
    print("Saved summary tables in the '" + OUTPUT_FOLDER + "' folder.")

    if not have_matplotlib:
        print("Chart skipped because matplotlib is not installed (pip install matplotlib).")
        return


    figure, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()
    chart_number = 0
    while chart_number < 4:
        sample = sample_types[chart_number]
        names = []
        values = []
        for line in sensitivity:
            if line["sample_type"] == sample and line["n_tested"] >= MIN_TESTED_FOR_CHART:
                names.append(line["antibiotic"])
                values.append(line["percent_sensitive"])
        axes[chart_number].bar(names, values, color="seagreen")
        axes[chart_number].set_title(sample, fontsize=10)
        axes[chart_number].set_ylim(0, 100)
        axes[chart_number].set_ylabel("% sensitive")
        axes[chart_number].tick_params(axis="x", labelrotation=60, labelsize=7)
        chart_number = chart_number + 1
    figure.suptitle("Antibiotic sensitivity by sample type")
    figure.tight_layout()
    figure.savefig(OUTPUT_FOLDER + "/sensitivity_chart.png", dpi=150)
    plt.close(figure)
    print("Saved chart: " + OUTPUT_FOLDER + "/sensitivity_chart.png")



records = load_records()
print("AMR data entry")
print("Type lab IDs or codes only, never patient names.")
print("Loaded " + str(len(records)) + " saved isolates from " + DATA_FILE)

while True:
    print("")
    print("Menu")
    print("  1. Add a new isolate")
    print("  2. View all entered isolates")
    print("  3. Delete an isolate")
    print("  4. Show summary on screen")
    print("  5. Save summary tables and chart to files")
    print("  6. Quit")
    choice = input("Choose 1-6: ")
    choice = choice.strip()

    if choice == "1":
        add_isolate(records)
    elif choice == "2":
        view_records(records)
    elif choice == "3":
        delete_isolate(records)
    elif choice == "4":
        show_summary(records)
    elif choice == "5":
        save_outputs(records)
    elif choice == "6":
        print("Goodbye. Your entries are saved.")
        break
    else:
        print("Please type a number from 1 to 6.")

AMR data entry
Type lab IDs or codes only, never patient names.
Loaded 0 saved isolates from private_data/amr_records.csv

Menu
  1. Add a new isolate
  2. View all entered isolates
  3. Delete an isolate
  4. Show summary on screen
  5. Save summary tables and chart to files
  6. Quit
Choose 1-6: 1

--- New isolate ---
Lab ID or code (NOT the patient's name): 1
Gender:
  1. Male
  2. Female
  3. Not recorded
Enter the number: 2
Sample date (DD/MM/YYYY, or just press Enter for today): 
Sample type:
  1. Urine
  2. Throat swab
  3. Pus
  4. Sputum
  5. Wound swab
  6. Nasal swab
  7. Stool
  8. Other
Enter the number: 1
Organism:
  1. Escherichia coli
  2. Klebsiella spp.
  3. Pseudomonas spp.
  4. Staphylococcus aureus
  5. Streptococcus spp.
  6. Proteus spp.
  7. Enterococcus spp.
  8. Acinetobacter spp.
  9. Citrobacter spp.
  10. Enterobacter spp.
  11. Other (type the name)
Enter the number: 2
ESBL producer? (y/n): n
Enter the sensitivity result for each antibiotic:
  ampicillin (